# Modelos con acoplamiento: teoría general y Casos 0-2 de Draft4

Este notebook funciona únicamente como **main**. Toda la lógica simbólica vive en módulos `.py` comentados.

El cálculo se divide en dos niveles:

1. formulación tensorial sin elegir métrica, $f(r)$ ni perfil escalar;
2. sustitución del ansatz, resolución de $f(r)$ y evaluación final explícita.

## 0. Cargar contexto y módulos

In [1]:
from pathlib import Path
from IPython.display import Markdown, display
from mc_core import CouplingContext
from mc_general import build_general_theory
from mc_case0 import build_case0, evaluate_btz_ansatz
from mc_case1 import build_case1, evaluate_case1_ansatz
from mc_case2 import build_case2, evaluate_case2_ansatz
from mc_export import export_results

BASE = Path.cwd()
ctx = CouplingContext(BASE / 'salidas')

## 1. Teoría general $L(g^{ab},R_{abcd},\phi,\nabla_a\phi)$

Se construyen los cuatro momentos, Palatini, términos de borde, ecuaciones de Euler y la identidad de Bianchi-Noether.

In [2]:
build_general_theory(ctx)
general_keys = [s.key for s in ctx.steps if s.group.startswith('Teoria general')]
ctx.show(general_keys)

#### Accion y variables independientes

Objeto: `ctx.steps[0]` / clave `general_action`

$$
S[g,\phi] = \kappa\int_V d^Dx\,\sqrt{-g}\,L(g^{ab},R_{abcd},\phi,u_a),\quad u_a\equiv\nabla_a\phi
$$

La conexion es Levi-Civita y el Riemann se toma completamente covariante.

#### Cuatro momentos del lagrangiano

Objeto: `ctx.steps[1]` / clave `general_momenta`

$$
(P^{abcd},\,M_{ab},\,J^a,\,F_\phi) = \left(\frac{\partial L}{\partial R_{abcd}},\,\frac{\partial L}{\partial g^{ab}},\,\frac{\partial L}{\partial u_a},\,\frac{\partial L}{\partial\phi}\right)
$$

En cada derivada parcial se mantienen fijos los otros tres argumentos.

#### Simetrias heredadas por el momento de curvatura

Objeto: `ctx.steps[2]` / clave `riemann_symmetries`

$$
P^{abcd} = -P^{bacd}=-P^{abdc}=P^{cdab},\qquad P^{a[bcd]}=0
$$

Solo la proyeccion con las simetrias del Riemann contribuye a \(P^{abcd}\,\delta R_{abcd}\).

#### Regla de la cadena funcional

Objeto: `ctx.steps[3]` / clave `delta_L`

$$
\delta L = M_{ab}\delta g^{ab}+P^{abcd}\delta R_{abcd}+F_\phi\delta\phi+J^a\nabla_a\delta\phi
$$

Para un escalar, \(\delta(\nabla_a\phi)=\nabla_a\delta\phi\).

#### Variacion de la medida

Objeto: `ctx.steps[4]` / clave `delta_measure`

$$
\delta\sqrt{-g} = -\frac12\sqrt{-g}\,g_{ab}\delta g^{ab}
$$

...

#### Variacion completa de la accion

Objeto: `ctx.steps[14]` / clave `full_variation`

$$
\delta S = \kappa\int_V d^Dx\sqrt{-g}\,[E_{ab}\delta g^{ab}+E_\phi\delta\phi+\nabla_a\Theta^a]
$$

#### Potencial de borde total

Objeto: `ctx.steps[15]` / clave `symplectic_potential`

$$
\Theta^a = \delta v_P^a+J^a\delta\phi
$$

#### Identidad de Bianchi-Noether off-shell

Objeto: `ctx.steps[16]` / clave `general_bianchi`

$$
2\nabla^aE_{ab}+E_\phi u_b = 0
$$

No se imponen ecuaciones de campo. Si \(E_\phi=0\), entonces \(\nabla^aE_{ab}=0\).

#### Caso con simetria de desplazamiento

Objeto: `ctx.steps[17]` / clave `shift_symmetric`

$$
F_\phi=0 = E_\phi=-\nabla_aJ^a,\qquad 2\nabla^aE_{ab}-(\nabla_aJ^a)u_b\equiv0
$$

#### Ecuaciones de campo generales

Objeto: `ctx.steps[18]` / clave `field_equations_general`

$$
(E_{ab},E_\phi) = (0,0)
$$

# 2. Casos I: formulación tensorial sin ansatz

Aquí $g_{ab}$ y $\phi$ permanecen arbitrarios. Todavía no se sustituyen $f(r)$ ni $\phi=p\varphi$.

## 2.1 Caso-0: sector puramente gravitatorio

In [3]:
build_case0(ctx)
case0_tensor_keys = [s.key for s in ctx.steps if s.group.endswith('::Caso-0') and s.group.startswith('Casos I')]
ctx.show(case0_tensor_keys)

#### Truncamiento del Draft4

Objeto: `ctx.steps[19]` / clave `case0_truncation`

$$
\alpha_{n\ge2}=\beta_{m\ge1}=\beta_0=\alpha_1 = 0
$$

El campo escalar desaparece de la accion y queda desacoplado.

#### Lagrangiano sobreviviente

Objeto: `ctx.steps[20]` / clave `case0_lagrangian`

$$
L_0 = R+\frac{2}{\ell^2}
$$

#### Accion bulk

Objeto: `ctx.steps[21]` / clave `case0_action`

$$
I_0 = \frac{1}{16\pi G}\int_M d^3x\sqrt{-g}\left(R+\frac{2}{\ell^2}\right)
$$

#### Momento de curvatura

Objeto: `ctx.steps[22]` / clave `case0_P`

$$
P_0^{abcd} = \frac12\left(g^{ac}g^{bd}-g^{ad}g^{bc}\right)
$$

#### Momento metrico

Objeto: `ctx.steps[23]` / clave `case0_M`

$$
M^{(0)}_{ab} = 2R_{ab}
$$

La derivada se toma a \(R_{abcd}\) covariante fijo; \(2/\ell^2\) no depende de la metrica.

...

#### Doble divergencia

Objeto: `ctx.steps[27]` / clave `case0_doubledivP`

$$
-2\nabla^m\nabla^nP^{(0)}_{amnb} = 0
$$

#### Ecuacion metrica del caso-0

Objeto: `ctx.steps[28]` / clave `case0_Eab`

$$
E^{(0)}_{ab} = R_{ab}-\frac12g_{ab}\left(R+\frac{2}{\ell^2}\right)=G_{ab}-\frac{1}{\ell^2}g_{ab}
$$

#### Ecuacion escalar del caso-0

Objeto: `ctx.steps[29]` / clave `case0_Ephi`

$$
E^{(0)}_\phi = 0
$$

No es una ecuacion dinamica: phi no aparece en la accion.

#### Bianchi-Noether especializado

Objeto: `ctx.steps[30]` / clave `case0_bianchi`

$$
2\nabla^aE^{(0)}_{ab} = 2\nabla^aG_{ab}-\frac{2}{\ell^2}\nabla^ag_{ab}=0
$$

#### Accion mejorada para Dirichlet

Objeto: `ctx.steps[31]` / clave `case0_boundary`

$$
I^{(0)}_{\mathrm{tot}} = I_0+\frac{1}{8\pi G}\int_{\partial M}d^2x\sqrt{|h|}\,K
$$

El termino GHY cancela el residuo \(-2\delta K\). Para AdS se suma \(-\frac{1}{8\pi G\ell}\int\sqrt{|h|}\).

## 2.2 Caso-1: Einstein-AdS más escalar cinético

In [4]:
build_case1(ctx)
case1_tensor_keys = [s.key for s in ctx.steps if s.group.endswith('::Caso-1') and s.group.startswith('Casos I')]
ctx.show(case1_tensor_keys)

#### Truncamiento del Draft4

Objeto: `ctx.steps[32]` / clave `case1_truncation`

$$
\alpha_{n\ge2}=\beta_{m\ge1}=\beta_0 = 0,\qquad \alpha_1\ne0
$$

#### Lagrangiano antes de imponer el ansatz

Objeto: `ctx.steps[33]` / clave `case1_lagrangian`

$$
L_1[g,\phi] = R+\frac{2}{\ell^2}-\alpha_1X,\qquad X\equiv u^au_a,\quad u_a\equiv\nabla_a\phi
$$

#### Accion bulk

Objeto: `ctx.steps[34]` / clave `case1_action`

$$
I_1 = \frac{1}{16\pi G}\int_Md^3x\sqrt{-g}\left(R+\frac{2}{\ell^2}-\alpha_1X\right)
$$

#### Momento de curvatura

Objeto: `ctx.steps[35]` / clave `case1_P`

$$
P_1^{abcd}=\frac{\partial L_1}{\partial R_{abcd}} = \frac12\left(g^{ac}g^{bd}-g^{ad}g^{bc}\right)
$$

#### Momento metrico

Objeto: `ctx.steps[36]` / clave `case1_M`

$$
M^{(1)}_{ab}=\frac{\partial L_1}{\partial g^{ab}} = 2R_{ab}-\alpha_1u_au_b
$$

La derivada se toma manteniendo fijos \(R_{abcd}\), \(u_a\) y \(\phi\).

...

#### Ecuacion metrica sin ansatz

Objeto: `ctx.steps[43]` / clave `case1_Eab`

$$
E^{(1)}_{ab} = G_{ab}-\frac{1}{\ell^2}g_{ab}-\alpha_1\left(u_au_b-\frac12g_{ab}X\right)
$$

#### Ecuacion escalar sin ansatz

Objeto: `ctx.steps[44]` / clave `case1_Ephi`

$$
E^{(1)}_\phi=F_\phi^{(1)}-\nabla_aJ_1^a = 2\alpha_1\Box\phi
$$

#### Bianchi-Noether off-shell del Caso-1

Objeto: `ctx.steps[45]` / clave `case1_bianchi`

$$
2\nabla^aE^{(1)}_{ab}+E^{(1)}_\phi u_b = 0
$$

#### Termino de frontera escalar

Objeto: `ctx.steps[46]` / clave `case1_boundary_scalar`

$$
n_a\Theta^a_{(\phi)} = -2\alpha_1(n^au_a)\,\delta\phi
$$

Se anula genericamente con Dirichlet \(\delta\phi|_{\partial M}=0\).

#### Accion mejorada para Dirichlet

Objeto: `ctx.steps[47]` / clave `case1_boundary_action`

$$
I^{(1)}_{\mathrm{tot}} = I_1+\frac{1}{8\pi G}\int_{\partial M}d^2x\sqrt{|h|}\,K
$$

No se requiere un contratermino escalar para el principio variacional; la renormalizacion on-shell de la rama lenta es un problema adicional.

## 2.3 Caso-2: acoplamiento derivativo $\beta_0$ con la curvatura

In [5]:
build_case2(ctx)
case2_tensor_keys = [s.key for s in ctx.steps if s.group.endswith('::Caso-2') and s.group.startswith('Casos I')]
ctx.show(case2_tensor_keys)

#### Truncamiento del Draft4

Objeto: `ctx.steps[48]` / clave `case2_truncation`

$$
\alpha_{n\ge2}=\beta_{m\ge1}=\alpha_1 = 0,\qquad \beta_0\ne0
$$

#### Lagrangiano antes de imponer el ansatz

Objeto: `ctx.steps[49]` / clave `case2_lagrangian`

$$
L_2[g,\phi] = R+\frac{2}{\ell^2}+\ell^2\beta_0\left(3R_{ab}u^au^b-RX\right),\quad X=u^au_a
$$

#### Accion bulk

Objeto: `ctx.steps[50]` / clave `case2_action`

$$
I_2 = \frac{1}{16\pi G}\int_Md^3x\sqrt{-g}\,L_2
$$

#### Tensor coeficiente de Ricci

Objeto: `ctx.steps[51]` / clave `case2_C`

$$
C^{ab} = g^{ab}+\ell^2\beta_0H^{ab},\qquad H^{ab}=3u^au^b-Xg^{ab}
$$

El sector de curvatura puede escribirse como \(C^{ab}R_{ab}\).

#### Momento de curvatura

Objeto: `ctx.steps[52]` / clave `case2_P`

$$
P_2^{abcd} = \frac14\left(C^{ac}g^{bd}-C^{ad}g^{bc}-C^{bc}g^{ad}+C^{bd}g^{ac}\right)
$$

...

#### Ecuacion escalar sin ansatz

Objeto: `ctx.steps[59]` / clave `case2_Ephi`

$$
E^{(2)}_\phi = -\nabla_aJ_2^a=-2\ell^2\beta_0\nabla_a\left[(3R^{ab}-Rg^{ab})u_b\right]
$$

#### Bianchi-Noether off-shell del Caso-2

Objeto: `ctx.steps[60]` / clave `case2_bianchi`

$$
2\nabla^aE^{(2)}_{ab}+E^{(2)}_\phi u_b = 0
$$

#### Termino de frontera escalar

Objeto: `ctx.steps[61]` / clave `case2_boundary_scalar`

$$
n_aJ_2^a\,\delta\phi = 2\ell^2\beta_0n_a(3R^{ab}-Rg^{ab})u_b\,\delta\phi
$$

Se anula genericamente con Dirichlet; sobre el ansatz diagonal se anula automaticamente.

#### Residuo metrico bajo Dirichlet

Objeto: `ctx.steps[62]` / clave `case2_boundary_metric`

$$
\Theta_{(g)}\big|_{\delta h=0} = -2\delta K+2\ell^2\beta_0X\,\delta K-3\ell^2\beta_0u^au^b\delta K_{ab}
$$

#### Accion mejorada para Dirichlet

Objeto: `ctx.steps[63]` / clave `case2_boundary_action`

$$
I^{(2)}_{\mathrm{tot}} = I_2+\frac{1}{16\pi G}\int_{\partial M}d^2x\sqrt{|h|}\left(2K-2\ell^2\beta_0XK+3\ell^2\beta_0K_{ij}D^i\phi D^j\phi\right)
$$

# 3. Casos II: sustitución y resolución completa del ansatz

Desde este punto se usa

$$ds^2=-f(r)d\tau^2+\frac{dr^2}{f(r)}+r^2d\varphi^2,$$

y, para los Casos 1 y 2, $\phi=p\varphi$.

## 3.1 Caso-0: solución BTZ

In [6]:
evaluate_btz_ansatz(ctx)
case0_ansatz_keys = [s.key for s in ctx.steps if s.group.endswith('::Caso-0') and s.group.startswith('Casos II')]
ctx.show(case0_ansatz_keys)

#### Ansatz circular estatico

Objeto: `ctx.steps[64]` / clave `ansatz_metric`

$$
ds^2 = -f(r)d\tau^2+\frac{dr^2}{f(r)}+r^2d\varphi^2
$$

#### Metrica y metrica inversa

Objeto: `ctx.steps[65]` / clave `ansatz_metric_matrix`

$$
(g_{ab},g^{ab}) = \left(\left[\begin{matrix}- f{\left(r \right)} & 0 & 0\\0 & \frac{1}{f{\left(r \right)}} & 0\\0 & 0 & r^{2}\end{matrix}\right],\,\left[\begin{matrix}- \frac{1}{f{\left(r \right)}} & 0 & 0\\0 & f{\left(r \right)} & 0\\0 & 0 & \frac{1}{r^{2}}\end{matrix}\right]\right)
$$

#### Momento de curvatura sobre el ansatz

Objeto: `ctx.steps[66]` / clave `case0_ansatz_P`

$$
\{P_0^{abcd}\}_{\mathrm{indep}}\big|_{g[f]} = \left\{P_0^{{\tau}{r}{\tau}{r}}=- \frac{1}{2},\;P_0^{{\tau}{\varphi}{\tau}{\varphi}}=- \frac{1}{2 r^{2} f{\left(r \right)}},\;P_0^{{r}{\varphi}{r}{\varphi}}=\frac{f{\left(r \right)}}{2 r^{2}}\right\}
$$

#### Momento metrico sobre el ansatz

Objeto: `ctx.steps[67]` / clave `case0_ansatz_M`

$$
M^{(0)}_{ab}\big|_{g[f]} = \left[\begin{matrix}\frac{\left(r \frac{d^{2}}{d r^{2}} f{\left(r \right)} + \frac{d}{d r} f{\left(r \right)}\right) f{\left(r \right)}}{r} & 0 & 0\\0 & \frac{- r \frac{d^{2}}{d r^{2}} f{\left(r \right)} - \frac{d}{d r} f{\left(r \right)}}{r f{\left(r \right)}} & 0\\0 & 0 & - 2 r \frac{d}{d r} f{\left(r \right)}\end{matrix}\right]
$$

#### Momento de gradiente escalar sobre el ansatz

Objeto: `ctx.steps[68]` / clave `case0_ansatz_J`

$$
J_0^a\big|_{g[f]} = \left[\begin{matrix}0\\0\\0\end{matrix}\right]
$$

...

#### Curvatura escalar BTZ

Objeto: `ctx.steps[86]` / clave `btz_R`

$$
R\big|_{\mathrm{BTZ}} = - \frac{6}{\ell^{2}}
$$

#### Einstein sobre la solucion BTZ

Objeto: `ctx.steps[87]` / clave `btz_einstein`

$$
G_{ab}\big|_{\mathrm{BTZ}} = \frac{1}{\ell^2}g_{ab}
$$

#### Verificacion final de las ecuaciones de campo

Objeto: `ctx.steps[88]` / clave `btz_field_equations`

$$
E^{(0)}_{ab}\big|_{\mathrm{BTZ}} = \left[\begin{matrix}0 & 0 & 0\\0 & 0 & 0\\0 & 0 & 0\end{matrix}\right]
$$

#### Forma de curvatura constante

Objeto: `ctx.steps[89]` / clave `btz_constant_curvature`

$$
R_{abcd}\big|_{\mathrm{BTZ}} = -\frac{1}{\ell^2}(g_{ac}g_{bd}-g_{ad}g_{bc})
$$

#### Invariante de Kretschmann

Objeto: `ctx.steps[90]` / clave `btz_kretschmann`

$$
R_{abcd}R^{abcd}\big|_{\mathrm{BTZ}} = \frac{12}{\ell^4}
$$

Se usa la forma de curvatura constante ya verificada; evita una contraccion de ocho bucles en cada corrida.

## 3.2 Caso-1: perfil angular y rama logarítmica

In [7]:
evaluate_case1_ansatz(ctx)
case1_ansatz_keys = [s.key for s in ctx.steps if s.group.endswith('::Caso-1') and s.group.startswith('Casos II')]
ctx.show(case1_ansatz_keys)

#### Sustitucion simultanea de metrica y escalar

Objeto: `ctx.steps[91]` / clave `case1_ansatz_data`

$$
(ds^2,\phi) = \left(-f(r)d\tau^2+\frac{dr^2}{f(r)}+r^2d\varphi^2,\;p\varphi\right)
$$

#### Gradiente escalar y contraccion cinetica

Objeto: `ctx.steps[92]` / clave `case1_ansatz_gradient`

$$
(u_a,u^a,X) = \left(\left[\begin{matrix}0\\0\\p\end{matrix}\right],\,\left[\begin{matrix}0\\0\\\frac{p}{r^{2}}\end{matrix}\right],\,\frac{p^{2}}{r^{2}}\right)
$$

#### Momento de curvatura sobre el ansatz

Objeto: `ctx.steps[93]` / clave `case1_ansatz_P`

$$
\{P_1^{abcd}\}_{\mathrm{indep}}\big|_{g[f],\phi=p\varphi} = \left\{P_1^{{\tau}{r}{\tau}{r}}=- \frac{1}{2},\;P_1^{{\tau}{\varphi}{\tau}{\varphi}}=- \frac{1}{2 r^{2} f{\left(r \right)}},\;P_1^{{r}{\varphi}{r}{\varphi}}=\frac{f{\left(r \right)}}{2 r^{2}}\right\}
$$

#### Momento metrico sobre el ansatz

Objeto: `ctx.steps[94]` / clave `case1_ansatz_M`

$$
M^{(1)}_{ab}\big|_{g[f],\phi=p\varphi} = \left[\begin{matrix}\frac{\left(r \frac{d^{2}}{d r^{2}} f{\left(r \right)} + \frac{d}{d r} f{\left(r \right)}\right) f{\left(r \right)}}{r} & 0 & 0\\0 & \frac{- r \frac{d^{2}}{d r^{2}} f{\left(r \right)} - \frac{d}{d r} f{\left(r \right)}}{r f{\left(r \right)}} & 0\\0 & 0 & - \alpha_{1} p^{2} - 2 r \frac{d}{d r} f{\left(r \right)}\end{matrix}\right]
$$

#### Momento de gradiente escalar sobre el ansatz

Objeto: `ctx.steps[95]` / clave `case1_ansatz_J`

$$
J_1^a\big|_{g[f],\phi=p\varphi} = \left[\begin{matrix}0\\0\\- \frac{2 \alpha_{1} p}{r^{2}}\end{matrix}\right]
$$

...

#### Escalar de Ricci de la rama logaritmica

Objeto: `ctx.steps[113]` / clave `case1_R_solution`

$$
R\big|_{f=f_{(1)}} = \frac{\alpha_{1} p^{2}}{r^{2}} - \frac{6}{\ell^{2}}
$$

#### Tensor de Ricci de la rama logaritmica

Objeto: `ctx.steps[114]` / clave `case1_Ricci_solution`

$$
R_{ab}\big|_{f=f_{(1)}} = \left[\begin{matrix}\frac{2 \left(- \ell^{2} \left(\alpha_{1} p^{2} \log{\left(\frac{r}{r_{0}} \right)} + \lambda\right) + r^{2}\right)}{\ell^{4}} & 0 & 0\\0 & \frac{2}{\ell^{2} \left(\alpha_{1} p^{2} \log{\left(\frac{r}{r_{0}} \right)} + \lambda\right) - r^{2}} & 0\\0 & 0 & \alpha_{1} p^{2} - \frac{2 r^{2}}{\ell^{2}}\end{matrix}\right]
$$

#### Invariante cuadratico de Ricci

Objeto: `ctx.steps[115]` / clave `case1_Ricci2_solution`

$$
R_{ab}R^{ab}\big|_{f=f_{(1)}} = \frac{\alpha_{1}^{2} p^{4}}{r^{4}} - \frac{4 \alpha_{1} p^{2}}{\ell^{2} r^{2}} + \frac{12}{\ell^{4}}
$$

#### Invariante de Kretschmann en tres dimensiones

Objeto: `ctx.steps[116]` / clave `case1_K_solution`

$$
R_{abcd}R^{abcd}\big|_{f=f_{(1)}} = \frac{3 \alpha_{1}^{2} p^{4}}{r^{4}} - \frac{4 \alpha_{1} p^{2}}{\ell^{2} r^{2}} + \frac{12}{\ell^{4}}
$$

Se usa la identidad tridimensional \(R_{abcd}R^{abcd}=4R_{ab}R^{ab}-R^2\).

#### Flujo escalar normal al borde radial

Objeto: `ctx.steps[117]` / clave `case1_boundary_ansatz`

$$
n^au_a\big|_{\phi=p\varphi,\,r=\mathrm{cte}} = 0
$$

El gradiente es puramente tangencial; el termino de frontera escalar se anula sobre este fondo.

## 3.3 Caso-2: perfil angular y rama racional

In [8]:
evaluate_case2_ansatz(ctx)
case2_ansatz_keys = [s.key for s in ctx.steps if s.group.endswith('::Caso-2') and s.group.startswith('Casos II')]
ctx.show(case2_ansatz_keys)

#### Sustitucion simultanea de metrica y escalar

Objeto: `ctx.steps[118]` / clave `case2_ansatz_data`

$$
(ds^2,\phi) = \left(-f(r)d\tau^2+\frac{dr^2}{f(r)}+r^2d\varphi^2,\;p\varphi\right)
$$

#### Gradiente escalar y contraccion cinetica

Objeto: `ctx.steps[119]` / clave `case2_ansatz_gradient`

$$
(u_a,u^a,X) = \left(\left[\begin{matrix}0\\0\\p\end{matrix}\right],\,\left[\begin{matrix}0\\0\\\frac{p}{r^{2}}\end{matrix}\right],\,\frac{p^{2}}{r^{2}}\right)
$$

#### Coeficiente de Ricci sobre el ansatz

Objeto: `ctx.steps[120]` / clave `case2_ansatz_C`

$$
C^{ab}\big|_{g[f],\phi=p\varphi} = \left[\begin{matrix}\frac{\beta_{0} \ell^{2} p^{2} - r^{2}}{r^{2} f{\left(r \right)}} & 0 & 0\\0 & \frac{\left(- \beta_{0} \ell^{2} p^{2} + r^{2}\right) f{\left(r \right)}}{r^{2}} & 0\\0 & 0 & \frac{2 \beta_{0} \ell^{2} p^{2} + r^{2}}{r^{4}}\end{matrix}\right]
$$

#### Momento de curvatura sobre el ansatz

Objeto: `ctx.steps[121]` / clave `case2_ansatz_P`

$$
\{P_2^{abcd}\}_{\mathrm{indep}}\big|_{g[f],\phi=p\varphi} = \begin{aligned}&P_2^{{\tau}{r}{\tau}{r}}=\frac{\beta_{0} \ell^{2} p^{2} - r^{2}}{2 r^{2}},\\[2pt]&P_2^{{\tau}{\varphi}{\tau}{\varphi}}=\frac{- \beta_{0} \ell^{2} p^{2} - 2 r^{2}}{4 r^{4} f{\left(r \right)}},\\[2pt]&P_2^{{r}{\varphi}{r}{\varphi}}=\frac{\left(\beta_{0} \ell^{2} p^{2} + 2 r^{2}\right) f{\left(r \right)}}{4 r^{4}}\end{aligned}
$$

#### Momento metrico sobre el ansatz

Objeto: `ctx.steps[122]` / clave `case2_ansatz_M`

$$
\{M^{(2)}_{aa}\}_{\rm diag}\big|_{g[f],\phi=p\varphi} = \begin{aligned}&M^{(2)}_{\tau\tau}=- \frac{\left(\beta_{0} \ell^{2} p^{2} \left(2 r \frac{d^{2}}{d r^{2}} f{\left(r \right)} - \frac{d}{d r} f{\left(r \right)}\right) - 2 r^{2} \left(r \frac{d^{2}}{d r^{2}} f{\left(r \right)} + \frac{d}{d r} f{\left(r \right)}\right)\right) f{\left(r \right)}}{2 r^{3}},\\[4pt]&M^{(2)}_{rr}=\frac{\beta_{0} \ell^{2} p^{2} \left(2 r \frac{d^{2}}{d r^{2}} f{\left(r \right)} - \frac{d}{d r} f{\left(r \right)}\right) - 2 r^{2} \left(r \frac{d^{2}}{d r^{2}} f{\left(r \right)} + \frac{d}{d r} f{\left(r \right)}\right)}{2 r^{3} f{\left(r \right)}},\\[4pt]&M^{(2)}_{\varphi\varphi}=\frac{\beta_{0} \ell^{2} p^{2} \left(r \frac{d^{2}}{d r^{2}} f{\left(r \right)} - 2 \frac{d}{d r} f{\left(r \right)}\right) - 2 r^{2} \frac{d}{d r} f{\left(r \right)}}{r}\end{aligned}
$$

...

#### Escalar de Ricci de la solucion racional

Objeto: `ctx.steps[141]` / clave `case2_R_solution`

$$
R\big|_{f=f_{(2)}} = \frac{2 \left(3 \beta_{0}^{2} \ell^{6} \lambda p^{4} - 10 \beta_{0}^{2} \ell^{4} p^{4} r^{2} - \beta_{0} \ell^{4} \lambda p^{2} r^{2} - 9 \beta_{0} \ell^{2} p^{2} r^{4} - 3 r^{6}\right)}{\ell^{2} \left(\beta_{0} \ell^{2} p^{2} + r^{2}\right)^{3}}
$$

#### Tensor de Ricci de la solucion racional

Objeto: `ctx.steps[142]` / clave `case2_Ricci_solution`

$$
\{R_{aa}\}_{\rm diag}\big|_{f=f_{(2)}} = \begin{aligned}&R_{\tau\tau}=- \frac{2 r^{2} \left(\ell^{2} \lambda - r^{2}\right) \left(- \beta_{0}^{2} \ell^{6} \lambda p^{4} + 4 \beta_{0}^{2} \ell^{4} p^{4} r^{2} + \beta_{0} \ell^{4} \lambda p^{2} r^{2} + 3 \beta_{0} \ell^{2} p^{2} r^{4} + r^{6}\right)}{\ell^{4} \left(\beta_{0} \ell^{2} p^{2} + r^{2}\right)^{4}},\\[4pt]&R_{rr}=\frac{2 \left(- \beta_{0}^{2} \ell^{6} \lambda p^{4} + 4 \beta_{0}^{2} \ell^{4} p^{4} r^{2} + \beta_{0} \ell^{4} \lambda p^{2} r^{2} + 3 \beta_{0} \ell^{2} p^{2} r^{4} + r^{6}\right)}{r^{2} \left(\ell^{2} \lambda - r^{2}\right) \left(\beta_{0} \ell^{2} p^{2} + r^{2}\right)^{2}},\\[4pt]&R_{\varphi\varphi}=\frac{2 r^{2} \left(\beta_{0} \ell^{4} \lambda p^{2} - 2 \beta_{0} \ell^{2} p^{2} r^{2} - r^{4}\right)}{\ell^{2} \left(\beta_{0} \ell^{2} p^{2} + r^{2}\right)^{2}}\end{aligned}
$$

#### Invariante cuadratico de Ricci

Objeto: `ctx.steps[143]` / clave `case2_Ricci2_solution`

$$
R_{ab}R^{ab}\big|_{f=f_{(2)}} = \frac{4 \left(3 \beta_{0}^{4} \ell^{12} \lambda^{2} p^{8} - 20 \beta_{0}^{4} \ell^{10} \lambda p^{8} r^{2} + 36 \beta_{0}^{4} \ell^{8} p^{8} r^{4} - 2 \beta_{0}^{3} \ell^{10} \lambda^{2} p^{6} r^{2} - 6 \beta_{0}^{3} \ell^{8} \lambda p^{6} r^{4} + 60 \beta_{0}^{3} \ell^{6} p^{6} r^{6} + 3 \beta_{0}^{2} \ell^{8} \lambda^{2} p^{4} r^{4} + 47 \beta_{0}^{2} \ell^{4} p^{4} r^{8} + 2 \beta_{0} \ell^{4} \lambda p^{2} r^{8} + 18 \beta_{0} \ell^{2} p^{2} r^{10} + 3 r^{12}\right)}{\ell^{4} \left(\beta_{0} \ell^{2} p^{2} + r^{2}\right)^{6}}
$$

#### Invariante de Kretschmann en tres dimensiones

Objeto: `ctx.steps[144]` / clave `case2_K_solution`

$$
R_{abcd}R^{abcd}\big|_{f=f_{(2)}} = \frac{4 \left(3 \beta_{0}^{4} \ell^{12} \lambda^{2} p^{8} - 20 \beta_{0}^{4} \ell^{10} \lambda p^{8} r^{2} + 44 \beta_{0}^{4} \ell^{8} p^{8} r^{4} - 2 \beta_{0}^{3} \ell^{10} \lambda^{2} p^{6} r^{2} + 10 \beta_{0}^{3} \ell^{8} \lambda p^{6} r^{4} + 60 \beta_{0}^{3} \ell^{6} p^{6} r^{6} + 11 \beta_{0}^{2} \ell^{8} \lambda^{2} p^{4} r^{4} + 47 \beta_{0}^{2} \ell^{4} p^{4} r^{8} + 2 \beta_{0} \ell^{4} \lambda p^{2} r^{8} + 18 \beta_{0} \ell^{2} p^{2} r^{10} + 3 r^{12}\right)}{\ell^{4} \left(\beta_{0} \ell^{2} p^{2} + r^{2}\right)^{6}}
$$

Se usa la identidad tridimensional \(R_{abcd}R^{abcd}=4R_{ab}R^{ab}-R^2\).

#### Flujo escalar normal al borde radial

Objeto: `ctx.steps[145]` / clave `case2_boundary_ansatz`

$$
n_aJ_2^a\big|_{\phi=p\varphi,\,r=\mathrm{cte}} = 0
$$

La corriente es puramente tangencial sobre la metrica diagonal.

## 4. Objetos SymPy reutilizables

In [9]:
display(Markdown('### Caso-0: solución y residuo final'))
display(ctx.objects['f_case0_final'] if 'f_case0_final' in ctx.objects else ctx.objects['metric_btz'])
display(ctx.objects['E_btz'])
display(Markdown(r'### Caso-1: solución y residuo final'))
display(ctx.objects['f_case1_solution'])
display(ctx.objects['E_case1_solution'])
display(Markdown(r'### Caso-2: solución racional y residuo final'))
display(ctx.objects['f_case2_solution'])
display(ctx.objects['E_case2_solution'])

### Caso-0: solución y residuo final

Matrix([
[lambda - r**2/ell**2,                              0,    0],
[                   0, ell**2/(-ell**2*lambda + r**2),    0],
[                   0,                              0, r**2]])

Matrix([
[0, 0, 0],
[0, 0, 0],
[0, 0, 0]])

### Caso-1: solución y residuo final

-alpha_1*p**2*log(r/r_0) - lambda + r**2/ell**2

Matrix([
[0, 0, 0],
[0, 0, 0],
[0, 0, 0]])

### Caso-2: solución racional y residuo final

r**2*(-ell**2*lambda + r**2)/(ell**2*(beta_0*ell**2*p**2 + r**2))

Matrix([
[0, 0, 0],
[0, 0, 0],
[0, 0, 0]])

## 5. Verificaciones exactas

In [10]:
assert ctx.checks and all(value == 0 for value in ctx.checks.values())
display(Markdown(f'**{len(ctx.checks)} verificaciones simbólicas: todas dieron cero.**'))
for key, value in ctx.checks.items():
    print(f'{key}: {value}')

**33 verificaciones simbólicas: todas dieron cero.**

diffeo_moment_identity: 0
general_bianchi: 0
case0_divP: 0
case0_doubledivP: 0
case0_bianchi: 0
case1_divP: 0
case1_doubledivP: 0
case1_bianchi: 0
case2_bianchi: 0
ansatz_bianchi: 0
btz_ricci: 0
btz_R: 0
btz_einstein: 0
btz_field_equations: 0
btz_constant_curvature: 0
btz_kretschmann: 0
case1_ansatz_box: 0
case1_integrate_f: 0
case1_f_solution: 0
case1_bianchi_ansatz: 0
case1_noether_ansatz: 0
case1_field_solution: 0
case1_scalar_solution: 0
case1_R_solution: 0
case1_boundary_ansatz: 0
case2_moment_identity: 0
case2_ansatz_Ephi: 0
case2_f_solution: 0
case2_bianchi_ansatz: 0
case2_noether_ansatz: 0
case2_field_solution: 0
case2_scalar_solution: 0
case2_boundary_ansatz: 0


## 6. Exportar el informe LaTeX/PDF

El informe conserva los dos niveles y muestra los momentos antes y después de sustituir cada solución.

In [11]:
export_info = export_results(ctx, compile_pdf=True)
export_info

{'tex': WindowsPath('C:/Investigacion/Simultations/ModelosConAcoplamiento/salidas/derivacion_modelos_con_acoplamiento_casos0_2.tex'),
 'pdf': WindowsPath('C:/Investigacion/Simultations/ModelosConAcoplamiento/salidas/derivacion_modelos_con_acoplamiento_casos0_2.pdf'),
 'checks': WindowsPath('C:/Investigacion/Simultations/ModelosConAcoplamiento/salidas/verificaciones_simbolicas.json'),
 'all_checks_zero': True,
 'check_count': 33,
 'compile_log': ' [5]\n[6] [7] [8] [9] [10] [11] [12] [13] [14] [15] [16] [17] [18] [19] [20] [21]\n[22] [23] [24] [25] [26] [27] [28]\n(derivacion_modelos_con_acoplamiento_casos0_2.aux) ) <C:\\Users\\guill\\AppData\\Lo\ncal\\MiKTeX\\fonts/pk/ljfour/jknappen/ec/dpi600\\ecbx1200.pk> <C:\\Users\\guill\\AppD\nata\\Local\\MiKTeX\\fonts/pk/ljfour/jknappen/ec/dpi600\\ectt1095.pk> <C:\\Users\\guil\nl\\AppData\\Local\\MiKTeX\\fonts/pk/ljfour/jknappen/ec/dpi600\\eccc1095.pk> <C:\\User\ns\\guill\\AppData\\Local\\MiKTeX\\fonts/pk/ljfour/jknappen/ec/dpi600\\ecbx1095.pk> <C